# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 4: Prompt Engineering</font>

# <font color="#003660">Notebook 2: Prompt Template Engineering</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know how a chat model is created from a base model and why it stops after its answer, <br>
        ... can write raw chat prompts with the special tokens of a chat template, including the system prompt, <br>
        ... understand how reasoning models write their reasoning between <code>&lt;think&gt;</code> and <code>&lt;/think&gt;</code>, <br>
        ... can find a model's template and look at the text that chat APIs build from your JSON messages.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [Raschka (2024): Building a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)
* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

These documentations can help you for your projects:
* Decoding strategies: [Hugging Face (2025): Generation strategies](https://huggingface.co/docs/transformers/generation_strategies)
* Chat templates: [Hugging Face: Chat templates](https://huggingface.co/docs/transformers/chat_templating)
* Ollama API: [Ollama API documentation](https://docs.ollama.com/api)
* OpenAI API: [OpenAI Python SDK](https://platform.openai.com/docs/api-reference/chat/create?lang=python)
* Ollama provides OpenAI API compatible endpoints: [Ollama](https://github.com/ollama/ollama), [Ollama Python](https://github.com/ollama/ollama-python), [Ollama OpenAI Compatibility](https://docs.ollama.com/api/openai-compatibility)
* HuggingFace provides OpenAI API compatible endpoints: [HuggingFace Inference Providers](https://huggingface.co/docs/inference-providers/index?python-clients=openai&javascript-clients=openai), [HuggingFace Endpoints](https://huggingface.co/docs/inference-endpoints/index)

## Where We Are

In [4.01 - Basics of Prompt Engineering](https://github.com/olivermueller/amlta-2026/blob/main/Session_04/4.01%20-%20Basics%20of%20Prompt%20Engineering.ipynb) you saw that a base model is a **continuation machine**: it continues zero-, one-, and few-shot patterns and chain-of-thought examples, and it does not stop on its own. A stop sequence could cut the text, but it is a crude fix.

In this notebook we look at the better fix: chat models, which were trained on one fixed **chat template** and end every answer with a special token. We will write these templates by hand, add a system prompt, and finally look at reasoning models.

## Setup

The same setup as in 4.01. If you work in a new Colab runtime, run all cells of this section again. On macOS or Windows with the Ollama app, skip Steps 1 and 2.

### Step 1: Install Ollama

In [1]:
!sudo apt-get install -y pciutils zstd

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

### Step 2: Start the Ollama server

In [3]:
import subprocess
import time

# the server writes its log into a file; a pipe that nobody reads could fill up and block the server
ollama_log = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)

# give the server a few seconds to start
time.sleep(5)

### Step 3: Pull the chat model

In 4.01 we used the base model `llama3.2:1b-text-q4_K_M`. Now we pull `llama3.2:1b`: the same model after additional training on conversations. The reasoning model follows in Part 3.

In [4]:
# You can work on a small gpu (e.g., T4) --> download will take a while (1 minute)
!ollama pull llama3.2:1b

In [5]:
CHAT_MODEL = "llama3.2:1b"  # chat model: Llama 3.2 1B after additional training on conversations

### Step 4: Connect from Python

As in 4.01, we use Ollama's own Python package, `ollama`.

In [6]:
!pip install ollama

In [7]:
import ollama

# address of our local Ollama server
OLLAMA_HOST = "http://127.0.0.1:11434"

# Ollama's own client, as in 4.01
olm = ollama.Client(host=OLLAMA_HOST)

## Part 1: Chat Models

### From base model to chat model

![Foundation Models VS Chat Models](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_04/imgs/llms.png)

(Image adapted from ([Raschka (2024)](https://www.manning.com/books/build-a-large-language-model-from-scratch)))

As visible in the image above, a **foundation model**, also called **base model**, is generated by *pretraining*, like the models you trained in the last session. After pretraining, you can fine-tune it on example conversations so that it follows instructions and answers in a conversational manner ([Ouyang et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/b1efde53be364a73914f58805a001731-Paper-Conference.pdf)). This is what was done with ChatGPT, Claude, and other models ([OpenAI, 2022](https://openai.com/index/chatgpt/); [Ganguli et al., 2022](https://doi.org/10.48550/arXiv.2209.07858)).

Our `CHAT_MODEL` is such a **chat model**: the same Llama 3.2 1B, fine-tuned on conversations. We will fine-tune models ourselves in Session 7.

### How the model sees it

During this fine-tuning, every conversation is written in one fixed format: the **chat template**. For Llama 3.2 it looks like this:

<pre>
<b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#1F5F9E">user</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#1F5F9E">What is the capital of France?</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b><b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#0B6048">assistant</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#0B6048">The capital of France is Paris.</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b>
</pre>

<b><font color="#B8860B">■ special token</font></b> &nbsp; <b><font color="#1F5F9E">■ user message</font></b> &nbsp; <b><font color="#0B6048">■ assistant (written by the model)</font></b> &nbsp; ↵ = line break

The parts:

* `<|start_header_id|>` and `<|end_header_id|>` enclose the **role**: `system`, `user`, or `assistant`. They are **special tokens**: each of them is one single token in the vocabulary, and they never appear in normal text.
* After the header come two line breaks and the message.
* `<|eot_id|>` ("end of turn") closes every message.
* To get an answer, the text ends with an **open assistant header**. The model continues from there. This is called the **generation prompt**.
* When the model has finished its answer, it generates `<|eot_id|>`, and Ollama stops. Ollama does not print the end token.
* Ollama also puts the start token `<|begin_of_text|>` in front of the text automatically, so we don't write it.

Let's write this by hand. But first, we need a new tool.

### New: raw mode

In 4.01 we sent our text with `olm.generate()`, and Ollama added nothing, because the template of a base model is only `{{ .Prompt }}`. A chat model has a real template: Ollama would wrap our hand-written text in a second set of special tokens. To write the template ourselves, we switch it off with `raw=True`. Then the model sees exactly our text, special tokens included.

The next cell defines two helpers that use raw mode:

* `complete()` sends a prompt with `raw=True` and prints our prompt in normal color and the continuation in green, with special tokens highlighted. At the end, it tells you how many tokens the model generated and why it stopped.
* `score()` works as in 4.01, but in raw mode: it fills the placeholder `{text}` with every example, sends the prompt, and counts the correct answers.

In [8]:
import re

GREEN, ORANGE, GREY, RESET = "\033[92m", "\033[33m", "\033[90m", "\033[0m"
SPECIAL_TOKENS = re.compile(r"(<\|[^|>]+\|>|</?think>|<start_of_turn>|<end_of_turn>)")


def highlight(text, color="", newlines=False):
    """Highlights special tokens (inverse video) and, if wanted, shows line breaks as ↵."""
    if newlines:
        text = text.replace("\n", "↵\n")
    text = SPECIAL_TOKENS.sub(lambda m: f"\033[7m{m.group(0)}\033[27m", text)
    return f"{color}{text}{RESET}"


def generate(prompt, model=CHAT_MODEL, max_tokens=80, temperature=0, seed=42, stop=None, **options):
    """Sends a raw prompt: no template, the model sees exactly this text and continues it."""
    options = {"num_predict": max_tokens, "temperature": temperature, "seed": seed, **options}
    if stop:
        options["stop"] = stop
    return olm.generate(model=model, prompt=prompt, raw=True, options=options)


def complete(prompt, model=CHAT_MODEL, max_tokens=80, temperature=0, seed=42, stop=None, newlines=False, **options):
    """Sends a raw prompt and prints it: our prompt in normal color, the continuation in green."""
    response = generate(prompt, model, max_tokens, temperature, seed, stop, **options)
    print(highlight(prompt, newlines=newlines), end="")
    if response.thinking:  # in case Ollama separated the reasoning of a reasoning model
        print(highlight("<think>\n" + response.thinking + "\n</think>\n\n", ORANGE, newlines), end="")
    print(highlight(response.response, GREEN, newlines))
    print(f"\n{GREY}[{response.eval_count or 0} tokens generated, stopped because of: {response.done_reason}]{RESET}")


def check(answer, expected):
    """Looks for the expected answer behind "The answer is" or, if the model did not write that, in the first line."""
    answer = answer.split("</think>")[-1]  # ignore the reasoning of reasoning models
    position = answer.lower().find("answer is")
    if position >= 0:
        answer = answer[position + len("answer is"):]
    first_line = next((line.strip() for line in answer.splitlines() if line.strip()), "")
    correct = re.search(rf"\b{re.escape(expected)}\b", first_line, re.IGNORECASE) is not None
    return correct, first_line


def score(template, examples, model=CHAT_MODEL, max_tokens=10, temperature=0, seed=42, stop=None, **options):
    """Fills {text} in the template with every example, sends it raw, and counts the correct answers."""
    print(f"{GREY}Prompt for the first example:{RESET}")
    print(highlight(template.replace("{text}", examples[0][0]), GREY) + "\n")
    correct, tokens = 0, 0
    for text, expected in examples:
        response = generate(template.replace("{text}", text), model, max_tokens, temperature, seed, stop, **options)
        ok, answer = check(response.response, expected)
        correct += ok
        tokens += response.eval_count or 0  # an empty prompt generates nothing
        mark = f"{GREEN}✓{RESET}" if ok else f"{ORANGE}✗{RESET}"
        print(f"{mark} expected: {expected:<10} model: {answer[:70]}")
    print(f"\n{correct}/{len(examples)} correct, {tokens} tokens generated")

Now let's send our hand-written chat prompt to the chat model:

In [9]:
chat_prompt = """<|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(chat_prompt, model=CHAT_MODEL, max_tokens=100, newlines=True)

This time, `complete()` reports `stop`: the model generated `<|eot_id|>` on its own.

**The model does not know that it is the assistant.** It still continues a text, exactly like the base model. The difference is that it was trained on thousands of texts in this format, in which an assistant answers helpfully and then writes `<|eot_id|>`. The generation stops at this token, so the model never gets to write the next user message. If we did not stop it, it would happily continue the conversation, like the base model in Part 3 of 4.01. The model *imitates* an assistant, because that is the most likely continuation of this text.

### Your turn

1. **Predict:** what happens if the text ends with an open **user** header instead of an assistant header?
2. **Predict:** what happens with two user messages after each other? What will the model answer, and where does it get the name from?
3. Write a conversation with two turns by hand: a question, an answer that you write yourself, and a follow-up question.
4. A real task, information extraction: see below.

In [10]:
open_user = """<|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of France is Paris.<|eot_id|><|start_header_id|>user<|end_header_id|>

"""
complete(open_user, model=CHAT_MODEL, max_tokens=60, newlines=True)

In [11]:
two_users = """<|start_header_id|>user<|end_header_id|>

My name is Anna.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is my name?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(two_users, model=CHAT_MODEL, max_tokens=60, newlines=True)

<details>
<summary><b>Explanation</b> (open after running the cells)</summary>

1. The model writes the **user's** message. It continues whatever role is open, because it only imitates a conversation and has no idea which side it is on.
2. Look for "Anna" in the answer: the model can use the name, because both messages are part of the same text. A chat model has no memory: everything it "remembers" is in the text it receives. The template does not forbid two user messages in a row; it is just an unusual text. Our small model is confused by it: it may use the name and still claim that it does not know it.
</details>

In [12]:
# TODO: write a conversation with two turns by hand
my_conversation = """"""
complete(my_conversation, model=CHAT_MODEL, max_tokens=100, newlines=True)

4. **A real task: information extraction.** Build a chat template that extracts the city in which an event takes place from the announcements below. Some of them mention more than one city.
    * Run the zero-shot template below and look at the model's answers.
    * Add an example as an earlier `user` and `assistant` turn, written by yourself. This is few-shot prompting inside a chat template. Does it help?

In [13]:
# eight short announcements with the city in which the event takes place
EVENTS = [
    ("Our team meeting next Monday will be held in Munich instead of Cologne.", "Munich"),
    ("The concert, originally planned for Leipzig, has been moved to Dresden.", "Dresden"),
    ("Join us for the summer party in Paderborn on Friday at 6 pm!", "Paderborn"),
    ("The workshop takes place in Frankfurt. Participants from Bonn can take the train.", "Frankfurt"),
    ("The award ceremony will be held in Dortmund, not in Bremen as announced last week.", "Dortmund"),
    ("Students from Hanover and Kassel will meet in Erfurt for the hackathon.", "Erfurt"),
    ("After long discussions, the board decided to hold the annual meeting in Nuremberg.", "Nuremberg"),
    ("The press conference was moved from Berlin to Potsdam at short notice.", "Potsdam"),
]

In [14]:
extraction_zero_shot = """<|start_header_id|>user<|end_header_id|>

In which city does the event take place? Answer with the city only.

{text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
score(extraction_zero_shot, EVENTS)

In [15]:
# TODO: add an example as an earlier user and assistant turn (few-shot inside the chat template)
extraction_few_shot = """"""
score(extraction_few_shot, EVENTS)

### The easy way: messages as JSON

You don't want to write special tokens by hand every time. Chat APIs take a **list of messages** in JSON format, each with a `role` and a `content`, and apply the template for you:

In [ ]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},  # roles: system, user, assistant
]

response = olm.chat(model=CHAT_MODEL, messages=messages, options={"temperature": 0, "seed": 42})
print(response.message.content)

What did the API send to the model? Ollama has a debugging feature for this, `_debug_render_only`: it builds the text from the messages without generating anything. Our last helper, `what_the_model_sees()`, uses it and prints the text with highlighted special tokens:

In [ ]:
import requests

def what_the_model_sees(model, messages, think=None, newlines=True):
    """Lets Ollama build the prompt from a list of messages, without generating anything."""
    payload = {"model": model, "messages": messages, "stream": False, "_debug_render_only": True}
    if think is not None:
        payload["think"] = think
    response = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=300)
    response.raise_for_status()
    text = response.json()["_debug_info"]["rendered_template"]
    print(highlight(text, newlines=newlines))

In [ ]:
what_the_model_sees(CHAT_MODEL, messages)

Ollama's template for Llama 3.2 always adds a `system` block with the line `Cutting Knowledge Date: December 2023`, even though we did not write a system message. A template can add text that nobody wrote. And here are the two user messages from above, built by the API:

In [ ]:
what_the_model_sees(CHAT_MODEL, [
    {"role": "user", "content": "My name is Anna."},
    {"role": "user", "content": "What is my name?"},
])

## Part 2: The System Prompt

The **system prompt** is a message with the role `system`. It comes first and contains general instructions that precede the task-specific details: a role, a tone, rules, or an output format ([Zhang et al., 2024](https://doi.org/10.48550/arXiv.2410.14826)). It is written by the developer of an application, not by the user.

### How the model sees it

<pre>
<b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#7B3FA0">system</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#7B3FA0">You are an AI chatbot giving sarcastic answers.</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b><b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#1F5F9E">user</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#1F5F9E">How are you?</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b><b><font color="#B8860B">&lt;|start_header_id|&gt;</font></b><b><font color="#0B6048">assistant</font></b><b><font color="#B8860B">&lt;|end_header_id|&gt;</font></b>↵
↵
<font color="#0B6048">Oh, just living the dream of answering questions all day.</font><b><font color="#B8860B">&lt;|eot_id|&gt;</font></b>
</pre>

<b><font color="#B8860B">■ special token</font></b> &nbsp; <b><font color="#7B3FA0">■ system prompt</font></b> &nbsp; <b><font color="#1F5F9E">■ user message</font></b> &nbsp; <b><font color="#0B6048">■ assistant (written by the model)</font></b> &nbsp; ↵ = line break

To the model, the system prompt is simply the first part of the text. During fine-tuning it learned to give this part special weight, but it is still text, not a rule that the model must obey.

In [ ]:
system_prompt = """<|start_header_id|>system<|end_header_id|>

You are an AI chatbot giving sarcastic answers.<|eot_id|><|start_header_id|>user<|end_header_id|>

How are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(system_prompt, model=CHAT_MODEL, max_tokens=100, newlines=True)

### Your turn: answers in Morse code

Write a system prompt that makes the model answer **only in Morse code** (dots, dashes, and slashes). Try it with the raw template first. Small models often need an example, so you can combine the system prompt with a one-shot example, as in Part 2 of 4.01.

In [ ]:
# TODO: write your system prompt
morse_prompt = """<|start_header_id|>system<|end_header_id|>

TODO: your system prompt<|eot_id|><|start_header_id|>user<|end_header_id|>

Hello, how are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
complete(morse_prompt, model=CHAT_MODEL, max_tokens=100, newlines=True)

**Second task: summaries in a fixed format.** Write a system prompt that makes the model summarize a text in **one sentence of at most 15 words**, starting with `Summary:`. There is no automatic score this time: read the outputs and judge them yourself. Does the model keep the format for all three texts?

In [ ]:
# three short (fictional) news texts
TEXTS = [
    "The city council of Paderborn has approved a new plan for bicycle lanes. Over the next three years, 25 kilometres "
    "of new lanes will be built, mainly on roads leading to the university and the main station. The project will cost "
    "about 12 million euros, half of which will be paid by the federal state. Local shop owners worry about losing "
    "parking spaces, while student groups welcomed the decision.",
    "A small software company from Hamburg has released a free app that helps farmers detect plant diseases. Farmers "
    "take a photo of a leaf, and the app suggests what the disease might be and how to treat it. The app works offline, "
    "which is important in rural areas with poor mobile coverage. In the first month, more than 40,000 people downloaded it.",
    "Researchers found that people who take short walks after meals have lower blood sugar levels than people who sit "
    "down. In the study, 30 participants walked for ten minutes after lunch and dinner for two weeks. The effect was "
    "strongest in the first hour after eating. The researchers say that even a few minutes of light movement can help.",
]

In [ ]:
# TODO: write your system prompt for the summaries
summary_template = """<|start_header_id|>system<|end_header_id|>

TODO: your system prompt<|eot_id|><|start_header_id|>user<|end_header_id|>

{text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
for text in TEXTS:
    response = generate(summary_template.replace("{text}", text), max_tokens=60)
    print("-", response.response.strip())

### The easy way

With the API, the system prompt is the first message in the list:

In [ ]:
messages = [
    {"role": "system", "content": "You are an AI chatbot giving sarcastic answers."},
    {"role": "user", "content": "How are you?"},
]
response = olm.chat(model=CHAT_MODEL, messages=messages, options={"temperature": 0, "seed": 42})
print(response.message.content)
what_the_model_sees(CHAT_MODEL, messages)

Compare the built text with your hand-written one: Ollama puts our system prompt after the `Cutting Knowledge Date` line, in the same `system` block.

If there is a message history, the messages should alternate between `user` and `assistant`:

In [ ]:
messages = [
    {"role": "system", "content": "You are an AI chatbot giving sarcastic answers."},
    {"role": "user", "content": "How are you?"},
    {"role": "assistant", "content": "I looooove being an AI chatbot! Always being asked the same questions over and over again. It's the best!"},
    {"role": "user", "content": "What do you love the most?"},
]
response = olm.chat(model=CHAT_MODEL, messages=messages, options={"temperature": 0, "seed": 42})
print(response.message.content)

In [ ]:
# TODO: your Morse code system prompt with olm.chat()

## Part 3: From Chain-of-Thought to Reasoning Models

### A new model

For this part we need a new model: [Qwen3.5 2B](https://ollama.com/library/qwen3.5). To give it the whole GPU memory, we first stop the Ollama server, which also unloads the Llama models, and start it again. On a Colab T4 GPU, both models also fit next to each other, so this step is optional there. These two cells are for Colab and Linux, where we started the server ourselves. If you use the Ollama app on macOS or Windows, skip them.

In [ ]:
# stop the Ollama server (this unloads all models from the GPU memory)
!pkill -f "ollama serve"
time.sleep(2)

# start it again, exactly as in Step 2
ollama_log = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
time.sleep(5)

In [ ]:
# about 2.7 GB, the download will take a moment
!ollama pull qwen3.5:2b

In [ ]:
REASONING_MODEL = "qwen3.5:2b"

### From chain-of-thought to reasoning models

In Part 3 of 4.01, the model wrote its reasoning steps because our examples showed it how. Modern **reasoning models** were trained to do this on their own: for problems where the answer can be checked, such as math or code, they were rewarded for correct answers, and they learned to write a long reasoning text before the answer ([DeepSeek-AI, 2025](https://arxiv.org/abs/2501.12948)). You need neither examples nor "think step by step".

The reasoning gets its own place in the template: between two **thinking tags**, `<think>` and `</think>`.

### Excursion: where to find a model's template

Every model family has its own template. Ollama stores it with the model, and `show()` returns it. This is the template of our Llama chat model, written in Go's template language:

In [ ]:
print(olm.show(CHAT_MODEL).template)

Can you find the parts from Part 1? `{{ .System }}` and `{{ .Content }}` are placeholders that Ollama fills with your messages, and `range ... .Messages` is a loop over all of them. Now the template of Qwen3.5:

In [ ]:
print(olm.show(REASONING_MODEL).template)

Only `{{ .Prompt }}`? For newer models, Ollama builds the text with a *renderer* that is written directly into Ollama, so `show()` does not reveal it. Our helper `what_the_model_sees()` works for every model, because it asks Ollama for the finished text:

In [ ]:
what_the_model_sees(REASONING_MODEL, [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "When I was 6 my sister was half my age. Now I'm 70. How old is my sister?"},
], think=True)

A complete Qwen3.5 conversation with reasoning looks like this:

<pre>
<b><font color="#B8860B">&lt;|im_start|&gt;</font></b><b><font color="#7B3FA0">system</font></b>↵
<font color="#7B3FA0">You are a helpful assistant.</font><b><font color="#B8860B">&lt;|im_end|&gt;</font></b>↵
<b><font color="#B8860B">&lt;|im_start|&gt;</font></b><b><font color="#1F5F9E">user</font></b>↵
<font color="#1F5F9E">When I was 6 my sister was half my age. Now I'm 70. How old is my sister?</font><b><font color="#B8860B">&lt;|im_end|&gt;</font></b>↵
<b><font color="#B8860B">&lt;|im_start|&gt;</font></b><b><font color="#0B6048">assistant</font></b>↵
<b><font color="#B8860B">&lt;think&gt;</font></b>↵
<font color="#C05A00">Okay, when I was 6, my sister was 3. So she is 3 years younger ...</font>↵
<b><font color="#B8860B">&lt;/think&gt;</font></b>↵
↵
<font color="#0B6048">Your sister is 67 years old.</font><b><font color="#B8860B">&lt;|im_end|&gt;</font></b>
</pre>

<b><font color="#B8860B">■ special token</font></b> &nbsp; <b><font color="#7B3FA0">■ system prompt</font></b> &nbsp; <b><font color="#1F5F9E">■ user message</font></b> &nbsp; <b><font color="#C05A00">■ reasoning (written by the model)</font></b> &nbsp; <b><font color="#0B6048">■ assistant (written by the model)</font></b> &nbsp; ↵ = line break

Compared with Llama, the tokens are different, but the idea is the same: `<|im_start|>` plus the role and a line break opens a message, and `<|im_end|>` closes it. And when thinking is switched on, the template ends with `<think>`: the model starts writing *inside* the thinking block.

### How the model sees it

Let's write it by hand and send it raw. Reasoning models should be used with the settings that their developers recommend. For Qwen3.5, this includes a temperature of 1.0: with a low temperature, reasoning models tend to repeat themselves in endless loops. The long reasoning also needs a larger context window (`num_ctx`).

In [ ]:
reasoning_prompt = """<|im_start|>user
When I was 6 my sister was half my age. Now I'm 70. How old is my sister?<|im_end|>
<|im_start|>assistant
<think>
"""
complete(reasoning_prompt, model=REASONING_MODEL, max_tokens=4096, temperature=1.0, num_ctx=8192)

Read the text between the tags: the model writes down its thoughts, checks intermediate results, and sometimes corrects itself. Then it closes the block with `</think>` and writes the answer, followed by `<|im_end|>`. All of this is generated token by token, exactly like every other text in this notebook. Every written thought becomes input for the next tokens, as in the loop from Session 3. This costs time: look at the number of generated tokens.

### Your turn

**Predict:** what happens if the assistant's part already contains an *empty* thinking block, `<think>\n\n</think>\n\n`?

In [ ]:
no_thinking_prompt = """<|im_start|>user
When I was 6 my sister was half my age. Now I'm 70. How old is my sister?<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
complete(no_thinking_prompt, model=REASONING_MODEL, max_tokens=500, temperature=1.0)

<details>
<summary><b>Explanation</b></summary>

The thinking block is already closed, so the most likely continuation is the answer itself. The model skips the reasoning. This is exactly what the template does when you switch thinking off.
</details>

**Your turn, continued: thinking on or off?** Below are the word problems from 4.01. The two templates differ only in the thinking block. The user message also asks the model to end with `The answer is`, so that `score()` finds the answer.

**Predict** first: which version scores better, and how many tokens will each version generate? With thinking on, we use only the first three problems, because every answer takes a few hundred tokens.

Then try: add `Think step by step.` to the thinking-off template. Does chain-of-thought in the answer help as much as thinking?

In [ ]:
# six word problems with their answers
WORD_PROBLEMS = [
    ("A baker makes 4 trays with 6 cookies on each tray. He sells 10 cookies. How many cookies are left?", "14"),
    ("Lisa has 3 boxes with 5 pencils each. She gives 4 pencils to her brother. How many pencils does she have now?", "11"),
    ("A bus has 20 passengers. At the first stop, 7 get off and 3 get on. How many passengers are on the bus now?", "16"),
    ("Tom reads 8 pages every day. How many pages does he read in one week?", "56"),
    ("A shirt costs 15 euros. Anna buys 2 shirts and pays with a 50 euro note. How much change does she get?", "20"),
    ("There are 12 eggs in a carton. Mia uses 5 eggs for a cake and 2 for breakfast. How many eggs are left?", "5"),
]

In [ ]:
thinking_off = """<|im_start|>user
{text} End your answer with 'The answer is' and the number.<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
score(thinking_off, WORD_PROBLEMS, model=REASONING_MODEL, max_tokens=500, temperature=1.0)

In [ ]:
thinking_on = """<|im_start|>user
{text} End your answer with 'The answer is' and the number.<|im_end|>
<|im_start|>assistant
<think>
"""
score(thinking_on, WORD_PROBLEMS[:3], model=REASONING_MODEL, max_tokens=4096, temperature=1.0, num_ctx=8192)

### The easy way

Ollama's chat API separates the two parts: the reasoning lands in `message.thinking`, the answer in `message.content`.

In [ ]:
messages = [{"role": "user", "content": "When I was 6 my sister was half my age. Now I'm 70. How old is my sister?"}]

# no temperature here: Ollama uses the settings recommended for Qwen3.5, including temperature 1.0
response = olm.chat(model=REASONING_MODEL, messages=messages, think=True, options={"num_ctx": 8192})
print(highlight(response.message.thinking, ORANGE))
print("-" * 40)
print(response.message.content)

And how does the reasoning go back into the model in the next turn? We send an earlier answer *with* its thinking and a follow-up question:

In [ ]:
what_the_model_sees(REASONING_MODEL, [
    {"role": "user", "content": "What is 12 x 12?"},
    {"role": "assistant", "thinking": "12 x 12 = 12 x 10 + 12 x 2 = 120 + 24 = 144.", "content": "144."},
    {"role": "user", "content": "And 13 x 13?"},
], think=True)

Look for the earlier answer: its thinking block is gone. The Qwen3.5 template keeps only the final answers of earlier turns and drops their reasoning, which saves a lot of tokens. Other model families handle this differently, which is one more reason to look at the template.

## Reality Check: Why It Is Called Prompt *Engineering*

Prompt engineering is called engineering because it is no exact science. What works depends on the model, the task, the wording, and even the formatting. You have seen this yourself: compare your scores from the *Your turn* parts with those of your neighbours. Same model, same examples, different prompts, different accuracy. [Sclar et al. (2024)](https://arxiv.org/abs/2310.11324) changed only the format of prompts, such as separators, spacing, and capitalization, and the accuracy of the same model on the same task changed by up to 76 percentage points. Surveys such as *The Prompt Report* ([Schulhoff et al., 2024](https://arxiv.org/abs/2406.06608)) list dozens of prompting techniques, and many of them lose their effect with the next model generation, because LLMs, and even SLMs, get better every month.

So prompting tricks are not the focus of this course. What stays is what you learned today: **how a prompt template is structured, what the model actually sees, and how it continues it.** You will need this in the following weeks again and again: for structured outputs, tool calling, agents, and when we fine-tune models ourselves.

## Questions?

Now is the time for questions! Is anything about templates, special tokens, roles, system prompts, or thinking unclear? Ask now, because the following sessions build on it.

## What You Learned

* A chat model still sees **one sequence of tokens** and continues it. The chat template is formatting around this text.
* A **chat model** was fine-tuned on one fixed **chat template**: special tokens mark where a message starts and ends and who is speaking (`<|start_header_id|>` … `<|end_header_id|>` and `<|eot_id|>` in Llama 3.2, `<|im_start|>` and `<|im_end|>` in Qwen3.5). The text ends with an open assistant header, and the model stops when it generates the end-of-turn token.
* The model **does not know that it is the assistant**. It imitates the continuation of a conversation, and it writes whatever role is open.
* The **system prompt** is just the first block of the template. A template can also add text that nobody wrote, like Llama's `Cutting Knowledge Date`.
* **Reasoning models** write their reasoning between `<think>` and `</think>`. The template decides whether thinking is switched on and whether earlier reasoning is sent back to the model.
* **Every model family has its own template.** Chat APIs build it from your JSON messages, and `_debug_render_only` shows you the result.
* A **prompt template** with a placeholder can be tested on examples with known answers: measure, don't guess.
* Prompting tricks come and go. Knowing what the model sees will stay useful.

In the next notebook, **4.03 - Prompting with the OpenAI-Compatible API**, you do all of this with the API that almost every model provider offers.